In [17]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import re

# --------------------------
# CONFIG
# --------------------------
IN_DIR = Path(r"C:\Drought\Regridding and data clipping\SM")
ESA_FILE = IN_DIR / "SM_monthly_India_0p05deg_2000-2023.nc"
GLDAS_FILE = IN_DIR / "GLDAS_NOAH_monthly_India_0p05deg_2000-2023.nc"
OUT_FILE = IN_DIR / "SM_percentiles_monthly_India_0p05deg_2000present.nc"

BASE_START = "2001-01"
BASE_END   = "2020-12"
MIN_YEARS_PER_MONTH = 10
PCT_SCALE = 100.0

# --------------------------
# helpers
# --------------------------
def to_month_end(da):
    """Ensure timestamps are month-end, sorted, unique."""
    t = pd.to_datetime(da["time"].values)
    mend = pd.PeriodIndex(t, freq="M").to_timestamp("M")
    da2 = da.assign_coords(time=mend)
    # drop duplicate stamps if any
    _, idx = np.unique(da2["time"].values, return_index=True)
    return da2.isel(time=np.sort(idx)).sortby("time")

def detect_units(v):
    u = str(getattr(v, "units", "")).strip()
    return re.sub(r"\s+", " ", u)

def gldas_layer_to_volumetric(da, thickness_m, rho_w=1000.0):
    """
    Convert GLDAS layer from kg m-2 -> m3 m-3 using theta = mass / (rho_w * thickness)
    If already volumetric, return as-is. If units missing, assume kg m-2.
    """
    units = detect_units(da).lower()
    if "kg" in units or units == "":
        return (da / (rho_w * thickness_m)).assign_attrs(units="m3 m-3")
    if "m3 m-3" in units or "m^3 m^-3" in units:
        return da
    # fallback
    return (da / (rho_w * thickness_m)).assign_attrs(units="m3 m-3")

def pct_from_baseline(x, b):
    """
    Empirical percentile for array x given baseline b using Hazen positions.
    x, b are 1-D vectors (time). Returns 0–1.
    """
    b = b[np.isfinite(b)]
    if b.size < MIN_YEARS_PER_MONTH:
        return np.full_like(x, np.nan, dtype=float)
    b_sorted = np.sort(b)
    n = b_sorted.size
    probs = (np.arange(1, n + 1) - 0.5) / n  # Hazen
    return np.interp(x, b_sorted, probs, left=probs[0], right=probs[-1])

def monthwise_percentiles(da, base_start, base_end, pct_scale=PCT_SCALE):
    """
    Compute calendar-month percentiles vs baseline for DataArray (time, lat, lon).
    Key trick: rename baseline time dim to avoid alignment ('time' vs 'tbase').
    """
    base = da.sel(time=slice(base_start, base_end))
    outs = []
    for m in range(1, 13):
        xm = da.where(da["time.month"] == m, drop=True)
        bm = base.where(base["time.month"] == m, drop=True).rename({"time": "tbase"})
        pm = xr.apply_ufunc(
            pct_from_baseline, xm, bm,
            input_core_dims=[["time"], ["tbase"]],
            output_core_dims=[["time"]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float],
            join="override",         # safe now because core dims differ
        )
        outs.append(pm)
    out = xr.concat(outs, dim="time").sortby("time")
    return out.clip(0.0, 1.0) * pct_scale

def clip_physical(da, vmin=0.0, vmax=0.6):
    return da.clip(vmin, vmax)

# --------------------------
# load ESA CCI (surface SM)
# --------------------------
print("Loading ESA CCI surface SM …")
esa = xr.open_dataset(ESA_FILE)
esa_sm = esa["sm"] if "sm" in esa else esa[[v for v in esa.data_vars][0]]
esa_sm = to_month_end(esa_sm)
if "m3 m-3" not in detect_units(esa_sm).lower():
    esa_sm = esa_sm.assign_attrs(units="m3 m-3")
esa_sm = clip_physical(esa_sm)

# --------------------------
# load GLDAS NOAH (layers)
# --------------------------
print("Loading GLDAS NOAH soil moisture layers …")
g = xr.open_dataset(GLDAS_FILE)
l0 = g.get("SoilMoi0_10cm_inst");  l1 = g.get("SoilMoi10_40cm_inst");  l2 = g.get("SoilMoi40_100cm_inst")
assert l0 is not None and l1 is not None and l2 is not None, "Missing GLDAS soil moisture layers."
l0 = to_month_end(gldas_layer_to_volumetric(l0, 0.10))
l1 = to_month_end(gldas_layer_to_volumetric(l1, 0.30))
l2 = to_month_end(gldas_layer_to_volumetric(l2, 0.60))
sm_rz = (0.10*l0 + 0.30*l1 + 0.60*l2).rename("sm_rz").assign_attrs(
    long_name="Root-zone soil moisture (0–100 cm), volumetric", units="m3 m-3"
)
sm_rz = clip_physical(sm_rz)

# --------------------------
# Align time
# --------------------------
common_time = np.intersect1d(esa_sm["time"].values, sm_rz["time"].values)
esa_sm = esa_sm.sel(time=common_time).chunk({"time": 36, "lat": 300, "lon": 300})
sm_rz  = sm_rz.sel(time=common_time).chunk({"time": 36, "lat": 300, "lon": 300})

# --------------------------
# Percentiles vs 2001–2020
# --------------------------
print("Computing ESA surface SM percentiles …")
sm_surf_pct = monthwise_percentiles(esa_sm, BASE_START, BASE_END).rename("sm_surf_pct")
sm_surf_pct = sm_surf_pct.assign_attrs(
    long_name="Surface soil moisture percentile (calendar-month, baseline 2001–2020)",
    units="percent", cell_methods="time: mean (monthly)",
    source_dataset="ESA CCI SSM COMBINED (monthly, 0.05°)"
)

print("Computing GLDAS root-zone SM percentiles …")
sm_rz_pct = monthwise_percentiles(sm_rz, BASE_START, BASE_END).rename("sm_rz_pct")
sm_rz_pct = sm_rz_pct.assign_attrs(
    long_name="Root-zone (0–100 cm) soil moisture percentile (calendar-month, baseline 2001–2020)",
    units="percent", cell_methods="time: mean (monthly)",
    source_dataset="GLDAS Noah (monthly, 0.05°)"
)

# optional dryness percentiles (higher = drier)
sm_surf_dpct = (PCT_SCALE - sm_surf_pct).rename("sm_surf_dry_pct").assign_attrs(
    long_name="Surface soil moisture dryness percentile (100 - percentile)", units="percent"
)
sm_rz_dpct = (PCT_SCALE - sm_rz_pct).rename("sm_rz_dry_pct").assign_attrs(
    long_name="Root-zone soil moisture dryness percentile (100 - percentile)", units="percent"
)

# --------------------------
# Save NetCDF
# --------------------------
print(f"Writing {OUT_FILE} …")
ds_out = xr.Dataset(
    data_vars=dict(
        sm_surf_pct=sm_surf_pct,
        sm_rz_pct=sm_rz_pct,
        sm_surf_dry_pct=sm_surf_dpct,
        sm_rz_dry_pct=sm_rz_dpct,
    ),
    attrs=dict(
        title="Soil Moisture Percentiles for India (0.05°)",
        summary="Calendar-month empirical percentiles (Hazen) vs 2001–2020 baseline for ESA CCI surface SM and GLDAS Noah root-zone (0–100 cm) SM.",
        conventions="CF-1.8", crs="EPSG:4326",
        spatial_resolution="0.05 degree", temporal_aggregation="monthly",
        history="Generated by make_sm_percentiles_v3.py",
        baseline=f"{BASE_START}..{BASE_END}",
        notes="GLDAS layers converted to volumetric m3 m-3 if provided as kg m-2.",
    ),
)
comp = dict(zlib=True, complevel=4)
encoding = {v: comp for v in ds_out.data_vars}
encoding.update({"lat": {"dtype": "float32"}, "lon": {"dtype": "float32"}})
ds_out.to_netcdf(OUT_FILE, encoding=encoding)
print("Done.")


Loading ESA CCI surface SM …
Loading GLDAS NOAH soil moisture layers …
Computing ESA surface SM percentiles …
Computing GLDAS root-zone SM percentiles …
Writing C:\Drought\Regridding and data clipping\SM\SM_percentiles_monthly_India_0p05deg_2000present.nc …
Done.
